In [2]:
# Loading the data #
################################################################################
merged <- readRDS('data/seurat_rds/all.rds')

# Extract raw and normalized expression data
raw_expr <- merged@assays[['SCT']]@counts
norm_expr <- merged@assays[['SCT']]@data

# If working with human genes, subset the expression data to human genes
human_genes <- TRUE  # Set to FALSE if you want to include non-human genes
prefix <- ifelse(human_genes, 'hg38-', 'mm10-')
genes <- getSpeciesGenes(raw_expr, prefix)
raw_expr <- raw_expr[genes, ]
norm_expr <- norm_expr[genes, ]

# Remove the prefix from gene names and convert to uppercase
gene_names <- gsub(prefix, "", rownames(raw_expr))
gene_names <- toupper(gene_names)
rownames(raw_expr) <- gene_names
rownames(norm_expr) <- gene_names

# Extract cell metadata
cell_meta <- merged@meta.data

In [3]:
# Create dummy spatial coordinates (not meaningful but required by Giotto)
image_info <- merged@images
coords <- image_info[[1]]@coordinates
for (i in 2:length(image_info)) {
    coords <- rbind(coords, image_info[[i]]@coordinates)
}

# Check if coordinates align with the normalized expression data
print(all(rownames(coords) == colnames(norm_expr)))

[1] TRUE


In [4]:
# Ensure that 'imagerow' and 'imagecol' are numeric
coords$imagerow <- as.numeric(coords$imagerow)
coords$imagecol <- as.numeric(coords$imagecol)

# Adjust the coordinates for Giotto
giotto_coords <- coords[, c('imagerow', 'imagecol')]
colnames(giotto_coords) <- c('row_pxl', 'col_pxl')
giotto_coords[, 2] <- -giotto_coords[, 2]

In [5]:
# Set instructions for Giotto
python_path <- "/home/uqqvo1/.conda/envs/MB/bin/python3"  # Specify your Python path if necessary
myinst <- createGiottoInstructions(save_plot = TRUE, show_plot = TRUE, save_dir = save_dir,python_path = python_path)
# Create the Giotto object
giotto <- createGiottoObject(raw_exprs = raw_expr,
                             norm_expr = norm_expr,
                             spatial_locs = giotto_coords,
                             instructions = myinst,
                             cell_metadata = cell_meta)


 external python path provided and will be used 
Consider to install these (optional) packages to run all possible Giotto commands for spatial analyses:  scran MAST smfishHmrf trendsceek SPARK multinet RTriangle FactoMiner
 Giotto does not automatically install all these packages as they are not absolutely required and this reduces the number of dependencies

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.0 GiB”


In [6]:
######### Loading gene sets ######
filter_genes <- giotto@gene_metadata[[1]]
load_SHH = T
load_gsea_v2 = T
gene_lists <- loadGeneSets(load_SHH = load_SHH, load_gsea_v2 = load_gsea_v2,
                           human=T) # defined in helpers.R
list_names <- names(gene_lists)
print(gene_lists) # Looks good!

New names:
• `` -> `...1`


$SHH.A
  [1] "TOP2A"    "UBE2C"    "NUF2"     "BIRC5"    "CDK1"     "KIF2C"   
  [7] "RRM2"     "PBK"      "TPX2"     "FAM64A"   "NDC80"    "PRC1"    
 [13] "MELK"     "SPAG5"    "CDKN3"    "MAD2L1"   "UBE2T"    "CENPF"   
 [19] "ZWINT"    "CDC20"    "NUSAP1"   "CCNA2"    "KIAA0101" "MLF1IP"  
 [25] "CENPM"    "KIF22"    "ECT2"     "TK1"      "TYMS"     "PTTG1"   
 [31] "RAD51AP1" "PLK1"     "SMC4"     "CCNB1"    "CCNB2"    "FANCI"   
 [37] "DTL"      "MXD3"     "KIAA1524" "CKAP2"    "H2AFZ"    "SGOL1"   
 [43] "CENPN"    "NEK2"     "AURKA"    "RACGAP1"  "LDHA"     "CENPK"   
 [49] "ORC6"     "HMGB1"    "TMPO"     "KPNA2"    "PCNA"     "MZT1"    
 [55] "DTYMK"    "NCAPG"    "KIF18B"   "SAE1"     "H2AFY"    "NMU"     
 [61] "RNASEH2A" "CKS1B"    "EZH2"     "FEN1"     "VRK1"     "FBXO5"   
 [67] "TIMELESS" "CENPH"    "TUBB6"    "TUBA1B"   "XRCC2"    "DNAJC9"  
 [73] "LMNB2"    "PSRC1"    "TUBB4B"   "FANCA"    "GMNN"     "CKS2"    
 [79] "DNMT1"    "RFC3"     "CDCA4"    "NCAPG2"   "CENPL"

In [7]:
####### Perform the enrichment ! ########
# See details on how to run here:
# http://spatialgiotto.rc.fas.harvard.edu/giotto_guide_to_cell_type_enrichment.html

marker_matrix <- makeSignMatrixPAGE(list_names, gene_lists)


In [8]:
# Need to look at overlap between the sig genes and the genes we have !
enrich_results <- runPAGEEnrich(giotto, marker_matrix, expression_values='normalized',
                              p_value=T, n_times=100,
                              return_gobject = F)

[1] "Warning, NA only has NA overlapping genes. Will be removed."
[1] "Warning,  only has  overlapping genes. Will be removed."


In [9]:
# Retrieving the per-spot enrichment scores #
enrich_scores <- as.data.frame( enrich_results$matrix )
rownames(enrich_scores) <- enrich_scores[,1]
enrich_scores <- enrich_scores[,!(colnames(enrich_scores) %in% c('cell_ID'))]
colnames(enrich_scores) <- str_c(colnames(enrich_scores), 'enrich scores', sep=' ')

In [10]:
# Replace inf values with max not inf #
for (coli in 1:ncol(enrich_scores)) {
  isinf <- is.infinite(enrich_scores[,coli])
  enrich_scores[isinf,coli] <- max(enrich_scores[isinf==F,coli])
}

# Saving the output #
out_name <- save_data_dir
run_bool <- c(load_SHH, load_gsea_v2)
run_state <- c('SHH', 'gsea_v2')
for (i in 1:length(run_state)) {
  if (i==1) {middle=''} else {middle='_'}
  if (run_bool[i]) {
    out_name <- paste(out_name, middle, run_state[i], sep='')
  }
}
out_name <- paste(out_name, '_scores.txt', sep='')
write.table(enrich_scores, out_name, sep='\t', quote=F)